In [1]:
import mrcfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Patch
import ipywidgets as widgets

In [2]:
### Define phantom volume sizes
z, y, x = 100, 512, 512 
pixel_size = 10 #Angstrom
phantom_tomo = np.zeros((x, y, z))


In [6]:
### Load the two proteins

path = "C:\\Users\\ruben\\OneDrive - Delft University of Technology\\Documenten\\Nanobiology 6\\ARP\\protein_files"

## Ribosomes - emd_2938
with mrcfile.open(f"{path}/emd_2938.map.gz", permissive=True) as mrc: 
    vol_rib = mrc.data.astype(np.float32).copy() # shape (z, y, x) 
    voxel_rib = mrc.voxel_size # Å per voxel, per axis 
    origin_rib = mrc.header.origin

## Cohesin - emd_52297
with mrcfile.open(f"{path}/emd_52297.map.gz", permissive=True) as mrc: 
    vol_coh = mrc.data.astype(np.float32).copy() # shape (z, y, x) 
    voxel_coh = mrc.voxel_size # Å per voxel, per axis 
    origin_coh = mrc.header.origin

In [7]:
print(f"The dim of ribosomes are {vol_rib.shape}, with voxelsize {voxel_rib}, origin at {origin_rib}")
print(f"That means that ribosome takes {np.array(vol_rib.shape)*max(voxel_rib.tolist())/pixel_size} pixels")
print(f"The dim of cohesin are {vol_coh.shape}, with voxelsize {voxel_coh}, origin at {origin_coh}")
print(f"That means that cohesin takes {np.array(vol_coh.shape)*max(voxel_coh.tolist())/pixel_size} pixels")

The dim of ribosomes are (300, 300, 300), with voxelsize (1.1, 1.1, 1.1), origin at (0.0, 0.0, 0.0)
That means that ribosome takes [33.00000072 33.00000072 33.00000072] pixels
The dim of cohesin are (512, 512, 512), with voxelsize (0.9024, 0.9024, 0.9024), origin at (0.0, 0.0, 0.0)
That means that cohesin takes [46.20288086 46.20288086 46.20288086] pixels


In [8]:
def place_particles(mask, n, exclusion_radius, margin=0, seed=None, reserve_radius=None):
    """
    Randomly place up to n particle centers on voxels where mask == 1.
    Centers are returned in the input array's axis order.

    exclusion_radius controls spacing within this placement group. If
    reserve_radius is provided, the returned mask excludes that distance
    around each new center, for placement of another particle group.
    """
    rng = np.random.default_rng(seed)
    free = mask.astype(bool, copy=True)
    shape = free.shape

    if margin > 0:
        free[:margin] = free[-margin:] = False
        free[:, :margin] = free[:, -margin:] = False
        free[:, :, :margin] = free[:, :, -margin:] = False

    r = int(np.ceil(exclusion_radius))
    zz, yy, xx = np.ogrid[-r:r + 1, -r:r + 1, -r:r + 1]
    ball = (zz**2 + yy**2 + xx**2) <= exclusion_radius**2

    centers = []
    for _ in range(n):
        idx = np.flatnonzero(free)
        if idx.size == 0:
            print(f"No free positions left after {len(centers)} particles")
            break

        center = np.unravel_index(rng.choice(idx), shape)
        centers.append(center)

        lo = [max(ci - r, 0) for ci in center]
        hi = [min(ci + r + 1, size) for ci, size in zip(center, shape)]
        blo = [low - (ci - r) for low, ci in zip(lo, center)]
        bhi = [start + (high - low) for start, high, low in zip(blo, hi, lo)]

        free[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]] &= ~ball[
            blo[0]:bhi[0], blo[1]:bhi[1], blo[2]:bhi[2]
        ]

    if reserve_radius is None:
        next_free = free
    else:
        next_free = mask.astype(bool, copy=True)
        reserve_r = int(np.ceil(reserve_radius))
        zz, yy, xx = np.ogrid[-reserve_r:reserve_r + 1, -reserve_r:reserve_r + 1, -reserve_r:reserve_r + 1]
        reserve_ball = (zz**2 + yy**2 + xx**2) <= reserve_radius**2

        for center in centers:
            lo = [max(ci - reserve_r, 0) for ci in center]
            hi = [min(ci + reserve_r + 1, size) for ci, size in zip(center, shape)]
            blo = [low - (ci - reserve_r) for low, ci in zip(lo, center)]
            bhi = [start + (high - low) for start, high, low in zip(blo, hi, lo)]

            next_free[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]] &= ~reserve_ball[
                blo[0]:bhi[0], blo[1]:bhi[1], blo[2]:bhi[2]
            ]

    return np.array(centers, dtype=int).reshape(-1, 3), next_free

In [15]:
### Define the places + orientations where proteins will be inserted
n_proteins = 100
distribution = 0.8
n_rib = int(n_proteins*distribution)
n_coh = int(n_proteins*(1-distribution))
size_rib = max(np.array(vol_rib.shape)*max(voxel_rib.tolist())/pixel_size)
size_coh = max(np.array(vol_coh.shape)*max(voxel_coh.tolist())/pixel_size)
possible_places = np.ones((x, y, z))

places_rib = []
places_coh = []

c_coh, free = place_particles(
    possible_places,
    n_coh,
    exclusion_radius=2 * size_coh,
    margin=int(np.ceil(size_coh)),
    reserve_radius=size_rib + size_coh,
)
c_rib, _ = place_particles(
    free,
    n_rib,
    exclusion_radius=2 * size_rib,
    margin=int(np.ceil(size_rib)),
)

print(f"Placed {len(c_rib)}/{n_rib} ribosomes and {len(c_coh)}/{n_coh} cohesin")
print(f"Sphere radii: ribosome={size_rib:.2f}, cohesin={size_coh:.2f} voxels")
rib_distances = np.linalg.norm(c_rib[:, None, :] - c_rib[None, :, :], axis=2)
coh_distances = np.linalg.norm(c_coh[:, None, :] - c_coh[None, :, :], axis=2)
cross_distances = np.linalg.norm(c_rib[:, None, :] - c_coh[None, :, :], axis=2)
assert np.all(rib_distances[np.triu_indices(len(c_rib), k=1)] > 2 * size_rib)
assert np.all(coh_distances[np.triu_indices(len(c_coh), k=1)] > 2 * size_coh)
assert np.all(cross_distances > size_rib + size_coh)

No free positions left after 18 particles
No free positions left after 12 particles
Placed 12/80 ribosomes and 18/19 cohesin
Sphere radii: ribosome=33.00, cohesin=46.20 voxels


In [ ]:
def plot_particle_slice(z_slice):
    fig, ax = plt.subplots(figsize=(8, 8))
    image = ax.imshow(
        phantom_tomo[:, :, z_slice].T,
        cmap="gray",
        origin="lower",
        extent=(0, x, 0, y),
        interpolation="nearest",
        vmin=0,
        vmax=max(float(np.max(phantom_tomo)), 1e-12),
    )
    ax.set_aspect("equal")
    ax.set_xlabel("X (voxels)")
    ax.set_ylabel("Y (voxels)")
    ax.set_title(f"Tomogram at Z = {z_slice}")
    fig.colorbar(image, ax=ax, label="Density")
    plt.show()


widgets.interact(
    plot_particle_slice,
    z_slice=widgets.IntSlider(
        value=phantom_tomo.shape[2] // 2,
        min=0,
        max=phantom_tomo.shape[2] - 1,
        step=1,
        description="Z slice",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=50, continuous_update=False, description='Z slice', max=99), Output()), …

In [ ]:
### Add each protein to the empty phantom_tomo
from scipy.ndimage import affine_transform, zoom
from scipy.spatial.transform import Rotation as R

rng = np.random.default_rng(42)

def rotate_volume_random(vol, order=1, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    rot = R.random(random_state=rng)
    matrix = rot.as_matrix()[::-1, ::-1]   # arrays are (z,y,x), Rotation uses (x,y,z)
    center = (np.array(vol.shape) - 1) / 2
    offset = center - matrix.T @ center
    return affine_transform(
        vol, matrix.T, offset=offset, order=order, mode="constant", cval=0.0
    )


def resize_protein_volume(vol, target_size):
    target_shape = np.maximum(1, np.rint(target_size).astype(int))
    return zoom(vol, target_shape / np.array(vol.shape), order=1)


def add_volume_at_center(target, source_zyx, center_xyz):
    source_xyz = source_zyx.transpose(2, 1, 0)
    start = np.asarray(center_xyz, dtype=int) - np.array(source_xyz.shape) // 2
    stop = start + np.array(source_xyz.shape)
    target[start[0]:stop[0], start[1]:stop[1], start[2]:stop[2]] += source_xyz


rib_target_size = np.array(vol_rib.shape) * max(voxel_rib.tolist()) / pixel_size
coh_target_size = np.array(vol_coh.shape) * max(voxel_coh.tolist()) / pixel_size

phantom_tomo.fill(0)
for center in c_rib:
    rotated_rib = rotate_volume_random(vol_rib, order=1, rng=rng)
    resized_rib = resize_protein_volume(rotated_rib, rib_target_size)
    add_volume_at_center(phantom_tomo, resized_rib, center)

for center in c_coh:
    rotated_coh = rotate_volume_random(vol_coh, order=1, rng=rng)
    resized_coh = resize_protein_volume(rotated_coh, coh_target_size)
    add_volume_at_center(phantom_tomo, resized_coh, center)

print(f"Resized ribosome volume: {tuple(np.rint(rib_target_size).astype(int))}")
print(f"Resized cohesin volume: {tuple(np.rint(coh_target_size).astype(int))}")
print(f"Inserted {len(c_rib)} ribosomes and {len(c_coh)} cohesin proteins")